In [1]:
# ==========================================
# 05_modeling_production_v2.py
# Improved Customer Churn Prediction Pipeline
# ==========================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    fbeta_score
)

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_csv('../data/processed/featured_churn.csv')

print(f"Dataset Shape: {df.shape}")
print("Target Distribution:\n", df['Churn'].value_counts(normalize=True))

# ==========================================
# 2. SPLIT FEATURES / TARGET
# ==========================================

X = df.drop('Churn', axis=1)
y = df['Churn']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# ==========================================
# 4. TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ==========================================
# 5. MODEL PIPELINE (IMPROVED)
# ==========================================

pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        random_state=42,
        class_weight='balanced_subsample',
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        n_jobs=-1
    ))
])

# ==========================================
# 6. HYPERPARAMETER TUNING
# ==========================================

param_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [8, 10, 15],
    'model__min_samples_leaf': [2, 3, 5]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

print("\nTraining model...")
grid.fit(X_train, y_train)

best_pipeline = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)

# ==========================================
# 7. BASELINE EVALUATION
# ==========================================

y_prob = best_pipeline.predict_proba(X_test)[:, 1]

y_pred_default = (y_prob >= 0.5).astype(int)

print("\n--- Default Threshold (0.5) ---")
print(classification_report(y_test, y_pred_default))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# ==========================================
# 8. THRESHOLD OPTIMIZATION (F0.5 OPTIMIZED)
# ==========================================

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f0_5_scores = []

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    f0_5_scores.append(fbeta_score(y_test, preds, beta=0.5))

f0_5_scores = np.array(f0_5_scores)

best_idx = np.argmax(f0_5_scores)
optimal_threshold = thresholds[best_idx]

print(f"\nOptimal Threshold (F0.5 optimized): {optimal_threshold:.3f}")

y_pred_opt = (y_prob >= optimal_threshold).astype(int)

print("\n--- Optimized Threshold Results ---")
print(classification_report(y_test, y_pred_opt))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))

# ==========================================
# 9. SAVE MODEL
# ==========================================

joblib.dump(best_pipeline, '../outputs/model/churn_pipeline.pkl')
print("\nModel saved successfully.")

Dataset Shape: (7032, 23)
Target Distribution:
 Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64

Training model...
Fitting 3 folds for each of 18 candidates, totalling 54 fits

Best Parameters: {'model__max_depth': 8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}

--- Default Threshold (0.5) ---
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1033
           1       0.53      0.79      0.63       374

    accuracy                           0.75      1407
   macro avg       0.72      0.76      0.72      1407
weighted avg       0.80      0.75      0.77      1407

ROC-AUC: 0.8354954936299962

Optimal Threshold (F0.5 optimized): 0.660

--- Optimized Threshold Results ---
              precision    recall  f1-score   support

           0       0.86      0.87      0.86      1033
           1       0.62      0.60      0.61       374

    accuracy                           0.80      1407
   macro avg       